In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import mpl_axes_aligner
from factor_analyzer.rotator import Rotator

In [ ]:
#Functions to make life simpler
def pca_func(pca_data, title_suffix=""):
    """Perform PCA and plot explained variance."""
    pca = PCA()
    pca_out = pca.fit_transform(pca_data)
    
    # --- Each plot gets its own figure ---
    plt.figure(figsize=(6, 4))
    plt.plot(pca.explained_variance_, marker='o')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance')
    plt.title(f'PCA Explained Variance{title_suffix}')
    
    # Print summary in terminal
    summary = pd.DataFrame({
        'Explained Variance Ratio': pca.explained_variance_ratio_,
        'Cumulative Explained Variance': pca.explained_variance_ratio_.cumsum()
    })
    print("\nExplained variance summary", title_suffix)
    print(summary)
    
    return pca, pca_out

In [ ]:
# There is no biplot function in sklearn, so we create a simple one ourselves
#function to produce biplot
def biplot(dfScores: pd.DataFrame, dfLoadings: pd.DataFrame, pca, title="Biplot", new_axes=True) -> None:
    """Create a PCA biplot."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Scores plot
    ax.scatter(dfScores.PC1.values, dfScores.PC2.values, s=5, color='b') # s = size
    if new_axes:
        explvar = 100 * pca.explained_variance_ratio_
        ax.set_xlabel(f"PC1 ({explvar[0]:.1f}% explained var.)", fontsize=10)
        ax.set_ylabel(f"PC2 ({explvar[1]:.1f}% explained var.)", fontsize=10)
    else:
        ax.set_xlabel("PC1", fontsize=10)
        ax.set_ylabel("PC2", fontsize=10)
    
    # Second axes for loadings
    ax2 = ax.twinx().twiny()
    font = {'color': 'g', 'weight': 'bold', 'size': 10}
    
    # Loading vectors
    for col in dfLoadings.columns.values:
        tipx = dfLoadings.loc['PC1', col]
        tipy = dfLoadings.loc['PC2', col]
        ax2.arrow(0, 0, tipx, tipy, color='r', alpha=0.5)
        ax2.text(tipx * 1.05, tipy * 1.05, col, fontdict=font,
                 ha='center', va='center')
    
    # Align axes and fix square aspect ratio
    mpl_axes_aligner.align.xaxes(ax, 0, ax2, 0, 0.5)
    mpl_axes_aligner.align.yaxes(ax, 0, ax2, 0, 0.5)
    ax.set_aspect('equal', adjustable='datalim')
    ax2.set_aspect('equal', adjustable='datalim')
    
    plt.title(title)
    plt.tight_layout()

In [ ]:
# Load Data
df = pd.read_csv('data/USA2016data.csv')
print(df.head())

In [ ]:
# Unscaled data
pca, pca_out = pca_func(df, " (Unscaled)")
dfScores = pd.DataFrame(pca_out, columns=[f'PC{i}' for i in range(1, df.shape[1] + 1)])
dfLoadings = pd.DataFrame(pca.components_, columns=df.columns, index=dfScores.columns)

# Match score scale
#scale_factor_1 = np.max(np.abs(dfScores.values)) / np.max(np.abs(dfLoadings.values))
#dfLoadings_scaled = dfLoadings * scale_factor_1
#biplot(dfScores, dfLoadings_scaled, pca, title="Biplot - Unscaled Data")
biplot(dfScores, dfLoadings, pca, title="Biplot - Unscaled Data")

In [ ]:
# Scaled data
dfs = StandardScaler().fit_transform(df)
pca, pca_out = pca_func(dfs, " (Scaled)")
dfScores = pd.DataFrame(pca_out, columns=[f'PC{i}' for i in range(1, df.shape[1] + 1)])
dfLoadings = pd.DataFrame(pca.components_, columns=df.columns, index=dfScores.columns)
biplot(dfScores, dfLoadings, pca, title="Biplot - Scaled Data")


In [ ]:
# PCA with rotation
k = 3  # The number of components to be considered
pca = PCA(n_components=k, svd_solver="full", random_state=0)
pca.fit(dfs)

loadings = pca.components_.T

# SS loadings / Proportion)
rotator = Rotator(method="varimax")#, normalize=True)
loadings_rot = rotator.fit_transform(loadings)
R = rotator.rotation_

load_rot_df = pd.DataFrame(loadings_rot, index=df.columns, columns=["PC1", "PC2", "PC3"])
# sq = load_rot_df**2
# ss = sq.sum(axis=0)
# prop = ss / load_rot_df.shape[0]
# cum = prop.cumsum()
# summary = pd.DataFrame([ss, prop, cum],
#                       index=["SS loadings", "Proportion Var", "Cumulative Var"])


In [ ]:
# Print "salient" rotated loadings
def pretty(df, cutoff):
     
    return df.where(df.abs() >= cutoff, other="")

In [ ]:
# We can select a cutoff for showing loadings
# Loadings that are larger (in absolute value) than cut,  will be printed:
cut = 1/(len(load_rot_df)**0.5)  # Sum of squared elements is 1. So, if all same size, each is 1/sqrt(number of variables)
# cut = 0.1    # Some arbitrary value
# print(cut)
print("\nRotated Loadings:\n", pretty(load_rot_df, cutoff=cut))
print("\nRotation Matrix:\n", pd.DataFrame(R, index=["PC1", "PC2", "PC3"], columns=["RC1", "RC2", "RC3"]))


In [ ]:
scores = pca_out[:,[1,2,3]]
scoresrot = scores.dot(R)
scores_rot_df = pd.DataFrame(scoresrot, columns=["PC1", "PC2", "PC3"])

